# Sample Data Load / ETL for Brazil DB Project

*Much of this code was generated by **Databricks Genie AI** using the following prompt:*

---
```
Can you please write code to take data from the files
listed in brazil_db.mini_world.raw_data and perform ETL
to fit them into the tables defined brazil_db.mini_world?
```
---

Prerequisite:
- Ensure access to 'raw_data' volume in catalog
- upload neccessary raw data files into volume

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from itertools import chain

In [0]:
# Region types from: https://www.aventuradobrasil.com/blog/brazils-ecosystems-amazon-rainforest-cerrado-atlantic-rainforest-pantanal-caatinga-pampas-mangroves-and-restingas/

# Create region types reference table
region_types_data = [
    ("Amazon Rainforest", "The Green Lungs of the World", "Vulnerable to deforestation and climate change"),
    ("Cerrado", "The source of Brazil's largest rivers", "Logging, mining, and other human activies"),
    ("Atlantic Rainforest", "The Forest With The World's Highest Biodiversity", "Deforestation"),
    ("Caatinga", "The Dry Savannah of the Northeast", "Rising temperatures and drought"),
    ("Pantanal", "One of the largest inland wetlands in the world", "climate change, illegal slash-and-burn agriculture, and the draining of land for cattle ranching"),
    ("Pampas", "Large Pastures", "Nothing"),
    ("Mangroves and Restingas", "Endangered Coastal Ecosystems", "Urbanization"),
    ("Cocais", "The Region of the Palm Trees", "Nothing"),
    ("Araucaria Forests", "Fertile Soil in Southern Brazil", "Deforestation")
]

schema_region_types = StructType([
    StructField("region_type", StringType(), False),
    StructField("description", StringType(), True),
    StructField("region_weaknesses", StringType(), True)
])

df_region_types = spark.createDataFrame(region_types_data, schema_region_types)

print(f"Region types created: {df_region_types.count()} rows")
df_region_types.write.mode('overwrite').saveAsTable('brazil_db.mini_world.region_types')
display(spark.read.table('brazil_db.mini_world.region_types'))

In [0]:
# Define Brazilian state mapping
state_mapping = [
    (1, "Acre", "Pantanal"),
    (2, "Amazonas", "Amazon Rainforest"),
    (3, "Amapá", "Cerrado"),
    (4, "Maranhão", "Atlantic Rainforest"),
    (5, "Mato Grosso", "Pantanal"),
    (6, "Pará", "Pampas"),
    (7, "Rondônia", "Mangroves and Restingas"),
    (8, "Roraima", "Cocais"),
    (9, "Tocantins", "Araucaria Forests"),
]

schema = StructType([
    StructField("region_id", IntegerType(), False),
    StructField("region_name", StringType(), True),
    StructField("region_type", StringType(), True)
])

df_regions = spark.createDataFrame(state_mapping, schema)

print(f"Regions created: {df_regions.count()} rows")
df_regions.write.mode('overwrite').saveAsTable('brazil_db.mini_world.regions')
display(spark.read.table('brazil_db.mini_world.regions'))

In [0]:
# Create sample region managers (this would typically come from another data source)
region_managers_data = [
    (1, "João", "Silva"),
    (2, "Maria", "Santos"),
    (3, "Pedro", "Oliveira"),
    (4, "Ana", "Costa"),
    (5, "Carlos", "Ferreira"),
    (6, "Juliana", "Almeida"),
    (7, "Ricardo", "Souza"),
    (8, "Fernanda", "Lima"),
    (9, "Roberto", "Martins")
]

schema_region_managers = StructType([
    StructField("region_id", IntegerType(), False),
    StructField("m_first_name", StringType(), True),
    StructField("m_last_name", StringType(), True)
])

df_region_managers = spark.createDataFrame(region_managers_data, schema_region_managers)

print(f"Region managers created: {df_region_managers.count()} rows")
df_region_managers.write.mode('overwrite').saveAsTable('brazil_db.mini_world.region_managers')
display(spark.read.table('brazil_db.mini_world.region_managers'))

In [0]:
# Define file paths
raw_data_path = "/Volumes/brazil_db/mini_world/raw_data/"
co2_file = f"{raw_data_path}brazil_co2.csv"
deforestation_file = f"{raw_data_path}brazil_deforestation.csv"
landusage_file = f"{raw_data_path}brazil_landusage.csv"

In [0]:
# Load CO2 data
df_co2_raw = spark.read.csv(co2_file, header=True, inferSchema=True)
df_co2 = df_co2_raw.select(
        F.col("year").cast("int").alias("year"),
        F.col("co2").cast("float").alias("co2_emission")
    ) \
    .filter(F.col("co2_emission").isNotNull())
print(f"CO2 data loaded: {df_co2.count()} rows")
display(df_co2.limit(5))

In [0]:

# Load deforestation data
df_deforestation_raw = spark.read.csv(deforestation_file, header=True, inferSchema=True)

# State columns to unpivot
state_cols = ["AC", "AM", "AP", "MA", "MT", "PA", "RO", "RR", "TO"]

state_region_id_mapping = {
    "AC": 1,
    "AM": 2,
    "AP": 3,
    "MA": 4,
    "MT": 5,
    "PA": 6,
    "RO": 7,
    "RR": 8,
    "TO": 9
}
mapping_expr = F.create_map([F.lit(x) for x in chain(*state_region_id_mapping.items())])

# Unpivot the deforestation data (wide to long format)
df_deforestation = df_deforestation_raw.selectExpr(
    "`Ano/Estados` as year",
    *[f"'{state}' as state, {state} as deforestation" for state in state_cols]
)

# Use stack to unpivot
stack_expr = f"stack({len(state_cols)}, " + ", ".join([f"'{state}', `{state}`" for state in state_cols]) + ") as (state, deforestation)"
df_deforestation = df_deforestation_raw.select(
    F.col("`Ano/Estados`").alias("year"),
    F.expr(stack_expr)
).filter(F.col("deforestation").isNotNull()) \
.withColumn("region_id", mapping_expr[F.col("state")])\
.select("region_id", "year", "deforestation")

print(f"Deforestation data loaded: {df_deforestation.count()} rows")
display(df_deforestation.limit(5))

In [0]:
from pyspark.sql import window as w
df_climate_markers = df_deforestation.join(df_co2, on="year", how="left")

deforest_window_spec = w.Window.partitionBy("year")

df_climate_markers = (
    df_climate_markers.withColumn("total_deforestation", F.sum("deforestation").over(deforest_window_spec))
    .withColumn("co2_emission_adj", F.col("co2_emission") * F.col("deforestation") / F.col("total_deforestation"))
    .drop("co2_emission")
    .select("year", "region_id", F.col("deforestation").cast(FloatType()), F.col("co2_emission_adj").cast(FloatType()).alias("co2_emission"))
    )
print(f"Climate markers data created: {df_climate_markers.count()} rows")
df_climate_markers.write.mode('overwrite').saveAsTable('brazil_db.mini_world.climate_markers')
display(spark.read.table('brazil_db.mini_world.climate_markers'))

In [0]:
# Get unique land use types from the data
land_use_types_list = df_landusage_long.select("class_level_1").distinct().collect()

# Create descriptions based on the land use type names
land_use_types_data = []
for row in land_use_types_list:
    land_use_type = row["class_level_1"]
    if land_use_type:
        # Create a basic description
        description = f"{land_use_type} land classification"
        land_use_types_data.append((land_use_type, description))

schema_land_use_types = StructType([
    StructField("land_use_type", StringType(), False),
    StructField("description", StringType(), True)
])

df_land_use_types = spark.createDataFrame(land_use_types_data, schema_land_use_types)

print(f"Land use types created: {df_land_use_types.count()} rows")
display(df_land_use_types.orderBy("land_use_type"))

In [0]:
# Load land usage data
df_landusage_raw = spark.read.csv(landusage_file, header=True, inferSchema=True)

# Filter for state-level data only (exclude municipality level)
df_landusage_filtered = df_landusage_raw.filter(
    (F.col("municipality").isNull()) & 
    (F.col("state_acronym").isNotNull()) &
    (F.col("state_acronym") != "")
)

print(f"Land usage data loaded (state-level): {df_landusage_filtered.count()} rows")
display(df_landusage_filtered.select("state", "state_acronym", "class_level_1", "1985", "2000", "2024").limit(5))

In [0]:
# Get year columns (from 1985 to 2024)
year_columns = [str(year) for year in range(1985, 2025)]

# Create stack expression to unpivot years
stack_size = len(year_columns)
stack_expr = f"stack({stack_size}, " + ", ".join([f"{year}, `{year}`" for year in year_columns]) + ") as (year, area_hectares)"

# Unpivot the data
df_landusage_long = df_landusage_filtered.select(
    "state_acronym",
    "class_level_1",
    F.expr(stack_expr)
).filter(F.col("area_hectares").isNotNull())

print(f"Land usage unpivoted: {df_landusage_long.count()} rows")
display(df_landusage_long.limit(10))

In [0]:
# Calculate total area per state and year
df_total_area = df_landusage_long.groupBy("state_acronym", "year") \
    .agg(F.sum("area_hectares").alias("total_area"))

# Join back and calculate percentage
df_landusage_pct = df_landusage_long.join(
    df_total_area,
    ["state_acronym", "year"],
    "left"
).withColumn(
    "percent",
    (F.col("area_hectares") / F.col("total_area") * 100)
).select(
    "state_acronym",
    "year",
    F.col("class_level_1").alias("land_use_type"),
    "percent"
)

print(f"Land usage with percentages: {df_landusage_pct.count()} rows")
display(df_landusage_pct.orderBy("state_acronym", "year", "land_use_type").limit(10))

In [0]:
# Join land usage with regions to get region_id
df_land_usage = df_landusage_pct.join(
    df_regions.select("region_id", "state_acronym"),
    df_landusage_pct["state_acronym"] == df_regions["state_acronym"],
    "left"
).select(
    "region_id",
    F.col("year").cast("int"),
    "land_use_type",
    "percent"
).filter(F.col("region_id").isNotNull())

print(f"Land usage prepared: {df_land_usage.count()} rows")
display(df_land_usage.orderBy("region_id", "year", "land_use_type").limit(10))